<a href="https://colab.research.google.com/github/sigvehaug/MLwPython/blob/master/course_1_accdata_HKB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About this Introduction to the Winter School on Deep Learning

An introduction on how to perform typical machine learning tasks with Python.

PD Dr. Sigve Haug, 2025. https://github.com/sigvehaug/MLwPython

This work is licensed under <a href="https://creativecommons.org/share-your-work/public-domain/cc0/">CC0</a>.

Duration: 45 min


## Load and inspect dataset (exploratory data analysis)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls -l drive/MyDrive/Courses/

In [ ]:
!ls -l ./drive/MyDrive/Courses/WS-DL-2024/

In [ ]:
import pandas as pd
dataset1 = './drive/MyDrive/Courses/WS-DL-2024/Data-Walk.csv'
dataset2 = './drive/MyDrive/Courses/WS-DL-2024/Data-Jump.csv'

movement1 = pd.read_csv(dataset1)
movement2 = pd.read_csv(dataset2)

# Add the category column "type" for data labeling

movement1['type'] = 'walk'
movement2['type'] = 'jump'

movement = pd.concat([movement1, movement2])
movement

In [ ]:
movement.info()

In [ ]:
movement[movement['type']=='walk'].describe()

In [ ]:
movement[movement['type']=='jump'].describe()

### Visualize

In [ ]:
import seaborn as sns
sns.scatterplot(data=movement, x='Linear Acceleration x (m/s^2)',
                y='Linear Acceleration z (m/s^2)', hue='type');

In [ ]:
sns.scatterplot(data=movement, x='Linear Acceleration x (m/s^2)',
                y='Linear Acceleration y (m/s^2)', hue='type');

In [ ]:
sns.scatterplot(data=movement, x='Linear Acceleration y (m/s^2)',
                y='Linear Acceleration z (m/s^2)', hue='type');

### Clean and split the data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
training = movement.drop(columns=['Time (s)'])
train, test = train_test_split(training, train_size=0.8)
print(train.shape, test.shape)
train

## Train and Evaluate Models

Here we try two different models and check for overfitting.

In [ ]:
x_train   = train.drop(columns='type')
y_train   = train['type']

x_test   = test.drop(columns='type')
y_test   = test['type']

#Chose and train the model
#model = LogisticRegression()
model = RandomForestClassifier(max_depth=12, random_state=0)
model.fit(x_train, y_train)

#print(confusion_matrix(train['type'],model.predict(X=train.drop(columns='type'))))
#print(confusion_matrix(test['type'], model.predict(X=test.drop(columns='type'))))
#print(classification_report(test['type'], model.predict(X=test.drop(columns='type'))))
#print(classification_report(train['type'], model.predict(X=train.drop(columns='type'))))

In [ ]:
model.predict(x_train)

In [ ]:
print('Performance of the inference on the training set')
print(confusion_matrix(y_train,model.predict(x_train)))
print(classification_report(y_train, model.predict(x_train)))

In [ ]:
print('Performance of the inference on the test set')
print(confusion_matrix(y_test, model.predict(x_test)))
print(classification_report(y_test, model.predict(x_test)))

In teams of 4, think of ways to improve your classification, do it and publish your accuracy [here](https://docs.google.com/forms/d/e/1FAIpQLSfTODp5uhppXiCnYwQdEGxaUQdshr2H6jdQ0sDE7oe5Be4ulw/viewform?usp=sf_link).  

# ML Exercise Regression

In many cases the scalar value of interest - dependent variable - is (or can be approximated as) linear combination of the independent variables.

In linear regression the estimator is searched in the form: $$\hat{y}(w, x) = w_0 + w_1 x_1 + ... + w_p x_p$$

The parameters $w = (w_1,..., w_p)$ and $w_0$ are designated as `coef_` and `intercept_` in `sklearn`.

Reference: https://scikit-learn.org/stable/modules/linear_model.html

In [ ]:
# Scikit-learn (formerly scikits.learn and also known as sklearn) is a free
# software machine learning library for the Python programming language.
# It features various classification, regression and clustering algorithms,
# and is designed to interoperate with the Python numerical and scientific
# libraries NumPy and SciPy. (from wiki)

from sklearn import linear_model
from sklearn.model_selection import train_test_split

# common visualization module
from matplotlib import pyplot as plt

# numeric module
import numpy as np
# data analysis module
import pandas as pd

%matplotlib inline

In [ ]:
def house_prices_dataset(return_df=False, price_max=400000, area_max=40000):
#  path = 'data/AmesHousing.csv'
  path = 'https://raw.githubusercontent.com/sigvehaug/MLwPython/master/data/AmesHousing.csv'
  df = pd.read_csv(path, na_values=('NaN', ''), keep_default_na=False)

  # Clean up the column names
  rename_dict = {k:k.replace(' ', '').replace('/', '') for k in df.keys()}
  df.rename(columns=rename_dict, inplace=True)

  # Select the columns to be used and make feature and target dataframe
  useful_fields = ['LotArea',
                  'Utilities', 'OverallQual', 'OverallCond',
                  'YearBuilt', 'YearRemodAdd', 'ExterQual', 'ExterCond',
                  'HeatingQC', 'CentralAir', 'Electrical',
                  '1stFlrSF', '2ndFlrSF','GrLivArea',
                  'FullBath', 'HalfBath',
                  'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
                  'Functional','PoolArea',
                  'YrSold', 'MoSold'
                  ]
  target_field = 'SalePrice'

  df.dropna(axis=0, subset=useful_fields+[target_field], inplace=True)

  cleanup_nums = {'Street':      {'Grvl': 0, 'Pave': 1},
                  'LotFrontage': {'NA':0},
                  'Alley':       {'NA':0, 'Grvl': 1, 'Pave': 2},
                  'LotShape':    {'IR3':0, 'IR2': 1, 'IR1': 2, 'Reg':3},
                  'Utilities':   {'ELO':0, 'NoSeWa': 1, 'NoSewr': 2, 'AllPub': 3},
                  'LandSlope':   {'Sev':0, 'Mod': 1, 'Gtl': 3},
                  'ExterQual':   {'Po':0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex':4},
                  'ExterCond':   {'Po':0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex':4},
                  'BsmtQual':    {'NA':0, 'Po':1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex':5},
                  'BsmtCond':    {'NA':0, 'Po':1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex':5},
                  'BsmtExposure':{'NA':0, 'No':1, 'Mn': 2, 'Av': 3, 'Gd': 4},
                  'BsmtFinType1':{'NA':0, 'Unf':1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ':5, 'GLQ':6},
                  'BsmtFinType2':{'NA':0, 'Unf':1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ':5, 'GLQ':6},
                  'HeatingQC':   {'Po':0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex':4},
                  'CentralAir':  {'N':0, 'Y': 1},
                  'Electrical':  {'':0, 'NA':0, 'Mix':1, 'FuseP':2, 'FuseF': 3, 'FuseA': 4, 'SBrkr': 5},
                  'KitchenQual': {'Po':0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex':4},
                  'Functional':  {'Sal':0, 'Sev':1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4, 'Min2':5, 'Min1':6, 'Typ':7},
                  'FireplaceQu': {'NA':0, 'Po':1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex':5},
                  'PoolQC':      {'NA':0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex':4},
                  'Fence':       {'NA':0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv':4},
                  }

  df_X = df[useful_fields].copy()
  df_X.replace(cleanup_nums, inplace=True)  # convert continous categorial variables to numerical
  df_Y = df[target_field].copy()

  # Convert to numpy arrays and return only rows with values below given maxima
  x = df_X.to_numpy().astype(np.float32)
  y = df_Y.to_numpy().astype(np.float32)

  if price_max>0:
    idxs = y<price_max
    x = x[idxs]
    y = y[idxs]

  if area_max>0:
    idxs = x[:,0]<area_max
    x = x[idxs]
    y = y[idxs]

  return (x, y, df) if return_df else (x,y)

In [ ]:
x, y, df = house_prices_dataset(return_df=True)
print(x.shape, y.shape)
df.head()

In [ ]:
plt.plot(x[:, 0], y, '.r')
plt.xlabel('Area / ft^2')
plt.ylabel('Price / USD');

In [ ]:
# Make train/test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

# Fit the model
reg = linear_model.LinearRegression()
reg.fit(x_train, y_train)

In [ ]:
# Evaluate MSE, MAD, and R2 on train and test datasets

# Prediction:
y_p_train = reg.predict(x_train)
y_p_test = reg.predict(x_test)

# mse
print('Train MSE = %5.2f' % np.std(y_train - y_p_train))
print('Test MSE = %5.2f' % np.std(y_test - y_p_test))
# mse
print('Train MAE = %5.2f' % np.mean(np.abs(y_train - y_p_train)))
print('Test MAE = %5.2f' % np.mean(np.abs(y_test - y_p_test)))
# R2
print('Train R2 = %5.2f' % reg.score(x_train, y_train))
print('Test R2 = %5.2f' % reg.score(x_test, y_test))

# Plot y vs predicted y for test and train parts
plt.figure(figsize=(10,10))
plt.plot(y_train, y_p_train, 'b.', label='Train')
plt.plot(y_test, y_p_test, 'r.', label='Test')

plt.plot([0], [0], 'w.')  # dummy to have origin
plt.xlabel('True Price')
plt.ylabel('Predicted Price')
#plt.gca().set_aspect('equal')
plt.legend()
plt.plot()

# The Rest of the Winter School

Now we have seen how to use the Python library scikit-learn for ML. Tomorrow and the rest of the week, you will learn about deep neural networks and use PyTorch to perform ML. TensorFlow is an alternative to PyTorch.